# 16 · Map's polygon layers, checked against an independent sort

## What this checks

**Map** draws several polygon layers together. Before drawing, it collects them and puts them in order, the largest
first. A smaller band then draws on top of a larger one, and stays visible. This check covers Map release 0.49.0.

That step was once its own widget, Collect polygon layers. It was retired into Map on 13 November 2026, and the step
moved into Map unchanged.

There is no geometry to compare here. So this checks a **rule**, not arithmetic.

**The size used to order them is not an area**, and that is worth pinning down. Fieldwork calls it the extent:

- **Every ring.** For every ring of a polygon, outer edges and holes alike, it takes the area that ring encloses.
- **Added up.** It adds them all up. Holes are added, not taken away.
- **Square degrees.** The result is in square degrees, with no projection.

So it is only a key for sorting. Fieldwork's own code makes the point. **Convex hull** and **Combined area** ask for
a UTM zone, because they report areas. This step asks for none, because it does not.

## How it is checked

We sort the same polygons again by the same rule, without Fieldwork's code. The reference takes each ring as a
polygon of its own, and asks **shapely**, the Python library for shapes, for its area. Shapely uses GEOS, the
geometry library behind most GIS software. A ring on its own has no hole to take away, so its area is always added.
That is the same quantity, reached by a different route.

The data are three walking-time corridors from the John Snow example, drawn on one Map.

## What should happen

1. **The order** of the polygons matches exactly the order the rule gives.
2. **Nothing is added, dropped or merged.** The same polygons go in and come out.
3. **The counts** of layers, polygons and polygons carrying a band match exactly.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["shapely"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_16_stack_polygons import run

report = run(Path.cwd() / "fixtures" / "stack-polygons-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["polygons"], "polygons in", report["layers"], "layers")
print(f"order agrees          {report['orderAgrees']}")
print(f"same polygons in/out  {report['samePolygonsInAndOut']}")
print(f"counts agree          {report['countsAgree']}")
print(f"ties in extent        {report['extentTies']}")
print()
for position, row in enumerate(report["extents"], start=1):
    print(f"  {position}. {row['id']:<28} {row['minutes']:>4} min   {row['extentSquareDegrees']:.3e} sq deg")

## The result on record

First recorded on 13 November 2026, against Map release 0.29.0. Rerun on 26 November 2026, against Map release
0.49.0: **Agrees**.

The three corridors came out in the same order as the independent sort, largest first. There were no ties. The same
three polygons went in and came out, and the counts matched.

The first record was made when Collect polygon layers retired into Map. The earlier result for that widget stands
as the record of its own release.

Running the cell above repeats the check on the same data.

## What a tie would mean

Two polygons with the same extent leave the order open. A sort does not have to keep equal items in a fixed order.
So the check reports the number of ties. With none, the expected order is the only order, and a passing result means
what it seems to.

With a tie, this check would still pass, while the drawing could differ from one run to the next. So that count
matters more than it looks.

## What this does not show

- That largest first is the right order to draw in.
- That a reader will not mistake the extent for a measure of ground. It has no projection and adds holes, so it is a
  key for sorting only. That risk is why the rule is stated in full here.